In [1]:
# Create the files directly in Google Colab
import os

# 1. Create a valid CSV
with open('valid_data.csv', 'w', encoding='utf-8') as f:
    f.write("id,name,age\n1,Alex,28\n2,Bob,35\n3,Charlie,22")

# 2. Create a truly empty CSV (0 bytes)
with open('empty_file.csv', 'w', encoding='utf-8') as f:
    pass  # This creates the file but writes nothing

# 3. Create a headers-only CSV
with open('headers_only.csv', 'w', encoding='utf-8') as f:
    f.write("id,name,age")

print("Files created successfully!")
# Verify the empty file size is 0
print(f"empty_file.csv size: {os.path.getsize('empty_file.csv')} bytes")

Files created successfully!
empty_file.csv size: 0 bytes


In [2]:
import csv
import os

def safe_read_csv(file_path):
    """
    Safely read a CSV file with comprehensive error handling.

    Args:
        file_path (str): Path to the CSV file

    Returns:
        dict: Success status, data, and metadata OR error details.
    """
    # 1. Check if file exists
    if not os.path.exists(file_path):
        return {
            'success': False,
            'error': f"File not found: {file_path}"
        }

    # 2. Check if file is empty (0 bytes)
    if os.path.getsize(file_path) == 0:
        return {
            'success': False,
            'error': f"File is empty: {file_path}"
        }

    # 3. Attempt to process the file content
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            reader = csv.DictReader(file)
            columns = reader.fieldnames

            # Check if headers exist
            if not columns:
                return {
                    'success': False,
                    'error': "CSV file has no headers"
                }

            rows = list(reader)

            # Check if there are data rows after the header
            if len(rows) == 0:
                return {
                    'success': False,
                    'error': "CSV file has headers but no data rows"
                }

            return {
                'success': True,
                'data': rows,
                'columns': columns,
                'row_count': len(rows)
            }

    except PermissionError:
        return {
            'success': False,
            'error': f"Permission denied: {file_path}"
        }
    except UnicodeDecodeError:
        return {
            'success': False,
            'error': f"File encoding error (UTF-8 required): {file_path}"
        }
    except Exception as e:
        # Catch-all for unexpected issues
        return {
            'success': False,
            'error': f"Unexpected error: {str(e)}"
        }

# --- TEST SETUP & EXECUTION ---
if __name__ == "__main__":
    # First, let's create the dummy files for testing
    with open("valid_data.csv", "w", encoding='utf-8') as f:
        f.write("id,name,age\n1,Alex,28\n2,Bob,35\n3,Charlie,22")

    with open("empty_file.csv", "w", encoding='utf-8') as f:
        pass # Creates an empty file

    with open("headers_only.csv", "w", encoding='utf-8') as f:
        f.write("id,name,age")

    # Define test cases
    test_files = [
        "valid_data.csv",
        "missing_file.csv",
        "empty_file.csv",
        "headers_only.csv"
    ]

    for test_file in test_files:
        print(f"\n--- Testing: {test_file} ---")
        result = safe_read_csv(test_file)

        if result['success']:
            print(f"✓ Success! Read {result['row_count']} rows.")
            print(f"  Columns: {result['columns']}")
        else:
            print(f"✗ Error: {result['error']}")


--- Testing: valid_data.csv ---
✓ Success! Read 3 rows.
  Columns: ['id', 'name', 'age']

--- Testing: missing_file.csv ---
✗ Error: File not found: missing_file.csv

--- Testing: empty_file.csv ---
✗ Error: File is empty: empty_file.csv

--- Testing: headers_only.csv ---
✗ Error: CSV file has headers but no data rows
